# 05 — Paper Figures

Produces the glacier-level map figures used in the paper: observed vs predicted melt (2017-2025) and future melt projection (2025-2065), for a selected set of representative glaciers.

All logic lives in `glacier_melt.visualise`; this notebook only loads data, defines which glaciers to plot, calls the plotting functions, and saves figures.

**Note:** `GLACIER_CONFIGS` below selects which glaciers appear in the figures and their display zoom/padding. These were chosen interactively by visually inspecting candidate glaciers (good size, good overlap performance, visually distinct location) - adjust to taste.

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from glacier_melt.evaluate import assign_pixels_to_glaciers
from glacier_melt.visualise import (
    plot_observed_vs_predicted_panel, plot_future_melt_panel,
)

PREDICTIONS_DIR = Path("../data/Predictions")
RESULTS_DIR = Path("../results")
RGI_PATH = Path("../data/rgi70_peru.geojson")
OUTPUT_DIR = Path("../results")
OUTPUT_DIR.mkdir(exist_ok=True)

## Load data

Loads the 2017 predictions (with MLP scores, for the observed-vs-predicted panel) and the 2025 simulation melt years (for the future projection panel).

In [ ]:
df_2017 = pd.read_parquet(
    PREDICTIONS_DIR / "predictions_full_peru_r3holdout.parquet"
)
df_2017 = df_2017.rename(columns={"prob_mlp_ae64": "mlp_score"})

df_melt = pd.read_parquet(RESULTS_DIR / "simulation_melt_years.parquet")

rgi_gdf = gpd.read_file(RGI_PATH)
df_2017["rgi_id"] = assign_pixels_to_glaciers(df_2017, rgi_gdf)

print(f"2017 pixels: {len(df_2017):,}")
print(f"2025 simulation pixels: {len(df_melt):,}")

## Select glaciers to plot

Six glaciers chosen to represent a range of sizes and locations across Peru, with good per-glacier overlap performance from `03_evaluation.ipynb`.

Each entry needs:
- `name`: display name (for reference, not shown on plot)
- `zoom`: contextily basemap zoom level (typically 12-14 depending on glacier size)
- `pad`: display padding around the glacier extent, in degrees
- `load_pad`: padding for the data load bounding box (>= pad, avoids edge artefacts)
- `lon_offset`, `lat_offset`: optional recentring if the glacier extent is asymmetric

**TODO: fill in the six RGI IDs and configs selected from the candidate list in `03_evaluation.ipynb` (sorted by area and overlap_pct).**

In [ ]:
GLACIER_CONFIGS = {
    'RGI2000-v7.0-G-16-02683': {
        'name': 'Nevado Jalahuana',
        'zoom': 14,
        'pad':  0.005,
        'load_pad': 0.02,
        'lon_offset': 0.002,
        'lat_offset': 0
    },
    'RGI2000-v7.0-G-16-02584': {
        'name': 'Nevado Hualca Hualca',
        'zoom': 14,
        'pad':  0.005,
        'load_pad': 0.02,
        'lon_offset': 0,
        'lat_offset': 0
    },
    'RGI2000-v7.0-G-16-01349': {
        'name': 'Nevado Cochas',
        'zoom': 14,
        'pad':  0.02,
        'load_pad': 0.03,
        'lon_offset': 0,
        'lat_offset': 0.002
    },
    'RGI2000-v7.0-G-16-01422': {
        'name': 'Nevado Caraz',
        'zoom': 14,
        'pad':  0.015,
        'load_pad': 0.04,
        'lon_offset': 0,
        'lat_offset': 0
    },
    'RGI2000-v7.0-G-16-02325': {
        'name': 'Nevado Quelccaya',
        'zoom': 14,
        'pad':  0.05,
        'load_pad': 0.08,
        'lon_offset': 0.017,
        'lat_offset': 0
    },
    'RGI2000-v7.0-G-16-00940': {
        'name': 'Nevado Yerupaja',
        'zoom': 14,
        'pad':  0.03,
        'load_pad': 0.04,
        'lon_offset': 0,
        'lat_offset': -0
    },
}

assert len(GLACIER_CONFIGS) == 6, "Fill in GLACIER_CONFIGS with 6 glaciers before running"

## Figure: Observed vs predicted melt, 2017-2025

In [ ]:
fig = plot_observed_vs_predicted_panel(df_2017, GLACIER_CONFIGS)
fig.savefig(
    OUTPUT_DIR / "glacier_observed_vs_predicted.png",
    dpi=200, bbox_inches="tight",
)
plt.show()

## Figure: Future melt projection, 2025-2065

In [ ]:
fig = plot_future_melt_panel(df_melt, df_2017, GLACIER_CONFIGS)
fig.savefig(
    OUTPUT_DIR / "glacier_future_melt.png",
    dpi=200, bbox_inches="tight",
)
plt.show()